## 面试问题

循环状态对象怎么设计：goal/history/scratchpad/step/budget？

## 回答主线

循环状态对象是整条循环的唯一真相源，最小 schema 含 goal/history/scratchpad/step/budget/done。四原则：不可变演进、可序列化、事实(history)与推测(scratchpad)分离、预算显式入状态。本 Notebook 对比「散落在全局变量的状态」与「集中不可变状态对象」，验证后者能 json 序列化、快照、round-trip 恢复，前者做不到。

## 真实案例

客服 agent 处理退款工单：goal=工单，history=已执行动作，scratchpad=对意图的推测，step/budget=预算记账。数据为教学状态，不代表真实工单系统。

In [1]:
import json  # 引入 json 以序列化循环状态。

initial_state = {  # 定义集中的循环状态对象最小 schema。
    "goal": "处理退款工单 T-100",  # 任务目标。
    "history": [],  # 已发生的动作与观察（事实）。
    "scratchpad": {"intent": "unknown"},  # 工作记忆中的推测。
    "step": 0,  # 已推进步数。
    "budget": 5,  # 剩余步数预算。
    "done": False,  # 终止标记。
}  # 结束状态定义。

print("状态字段:", list(initial_state.keys()))  # 展示状态包含的字段。
print("初始状态:", initial_state)  # 展示完整初始状态。

状态字段: ['goal', 'history', 'scratchpad', 'step', 'budget', 'done']
初始状态: {'goal': '处理退款工单 T-100', 'history': [], 'scratchpad': {'intent': 'unknown'}, 'step': 0, 'budget': 5, 'done': False}


## 基线（Baseline）

反面基线：把状态散落到多个全局变量。它能推进，但状态不集中、无法整体快照或序列化。

In [2]:
g_step = 0  # 散落的步数变量。
g_history = []  # 散落的历史变量。
g_intent = "unknown"  # 散落的意图变量。

def scattered_advance(action, observed_intent):  # 用散落全局变量推进的反面写法。
    global g_step  # 声明修改全局步数。
    global g_intent  # 声明修改全局意图。
    g_step += 1  # 推进步数。
    g_history.append(action)  # 追加历史。
    g_intent = observed_intent  # 覆盖意图。

scattered_advance("classify", "refund")  # 执行一步散落推进。
print("散落状态 step:", g_step, "history:", g_history, "intent:", g_intent)  # 展示状态散落在多个变量。

散落状态 step: 1 history: ['classify'] intent: refund


## 核心实现：集中不可变状态 + 序列化

用一个函数不可变地推进状态（深拷贝后修改副本），并保证整个状态可 json 序列化。history 记录事实，scratchpad 记录推测，两者分开。

In [3]:
def advance(state, action, observed_intent):  # 不可变推进：返回新状态而非修改旧状态。
    new_state = json.loads(json.dumps(state))  # 深拷贝出一个新状态确保不可变。
    new_state["step"] = state["step"] + 1  # 推进步数。
    new_state["history"] = state["history"] + [action]  # 追加历史事实。
    new_state["scratchpad"] = {"intent": observed_intent}  # 更新推测而不写入 history。
    new_state["budget"] = state["budget"] - 1  # 扣减预算。
    return new_state  # 返回新状态。

In [4]:
s0 = initial_state  # 记录初始快照。
s1 = advance(s0, "classify", "refund")  # 执行第一步。
s2 = advance(s1, "verify_amount", "refund")  # 执行第二步。
print("s0 step/budget:", s0["step"], s0["budget"])  # 展示初始快照未被修改。
print("s1 step/budget:", s1["step"], s1["budget"])  # 展示第一步后状态。
print("s2 step/budget:", s2["step"], s2["budget"])  # 展示第二步后状态。
serialized = json.dumps(s2)  # 把状态序列化为字符串。
restored = json.loads(serialized)  # 从字符串恢复状态。
print("序列化长度:", len(serialized))  # 展示状态可完整序列化。
print("round-trip 一致:", restored == s2)  # 展示反序列化后与原状态相等。
print("history 演化:", s0["history"], "->", s1["history"], "->", s2["history"])  # 展示历史逐步追加。

s0 step/budget: 0 5
s1 step/budget: 1 4
s2 step/budget: 2 3
序列化长度: 171
round-trip 一致: True
history 演化: [] -> ['classify'] -> ['classify', 'verify_amount']


## 结果解读

初始快照 s0 未被后续步修改（step 仍为 0），说明不可变演进成立；s2 能完整 json 序列化并 round-trip 恢复；history 逐步追加事实，scratchpad 单独存推测。集中状态因此可快照、可持久化、可 diff。

## 失败案例与修正

进程重启后散落的全局变量被重置或覆盖，无法从任一「快照」完整恢复；而集中状态早已序列化，可从字符串完整复原（含 history 与预算）。

In [5]:
snapshot_step = g_step  # 尝试对散落状态做快照只能抓到单个变量。
g_step = 0  # 模拟进程重启后全局变量被重置。
g_history.clear()  # 历史也被清空。
print("散落状态重启后 step:", g_step, "history:", g_history, "-> 无法完整恢复")  # 展示散落状态丢失。
recovered = json.loads(serialized)  # 集中状态可从序列化完整恢复。
print("集中状态恢复 step/budget:", recovered["step"], recovered["budget"])  # 展示集中状态可完整恢复。
print("集中状态恢复 history:", recovered["history"])  # 展示历史也一并恢复。

散落状态重启后 step: 0 history: [] -> 无法完整恢复
集中状态恢复 step/budget: 2 3
集中状态恢复 history: ['classify', 'verify_amount']


In [6]:
assert s0["step"] == 0  # 不可变推进不应修改初始状态。
assert s2["step"] == 2  # 两步后步数应为 2。
assert s2["budget"] == 3  # 两步后预算应扣到 3。
assert restored == s2  # 序列化 round-trip 应保持一致。
assert recovered["history"] == ["classify", "verify_amount"]  # 集中状态应能完整恢复历史。
assert "intent" in s2["scratchpad"]  # 推测应保存在 scratchpad 而非 history。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
